## chunking

In [ ]:
import json
import re
from typing import List, Dict, Tuple
from dataclasses import dataclass
import uuid

@dataclass
class Chunk:
    """Chunk veri yapısı"""
    id: str
    content: str
    metadata: Dict
    chunk_type: str
    
class LegalDocumentChunker:
    """
    Hukuki RAG için optimize edilmiş chunking sistemi
    
    Temel Prensipler:
    1. Anlamsal bütünlük: Her chunk kendi başına anlamlı olmalı
    2. Bağlamsal yeterlilik: Soru cevap için yeterli bilgi içermeli
    3. Minimal parçalama: Sadece gerektiğinde böl
    4. Header propagation: Her chunk'ta doküman kimliği olmalı
    """
    
    def __init__(self, 
                 max_chunk_size: int = 3000,      # RAG için daha büyük chunk'lar
                 overlap_size: int = 300,          # Daha fazla overlap
                 min_chunk_size: int = 500):       # Çok küçük chunk'ları önle
        self.max_chunk_size = max_chunk_size
        self.overlap_size = overlap_size
        self.min_chunk_size = min_chunk_size
        
    def chunk_documents(self, json_file_path: str) -> List[Chunk]:
        """JSON dosyasındaki tüm dokümanları chunk'lara böl"""
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        all_chunks = []
        
        for doc in data:
            file_name = doc.get('file_name', 'unknown')
            content = doc.get('content', '')
            
            # Doküman tipini belirle
            doc_type = self._identify_document_type(file_name, content)
            
            # Tipe göre chunking yap
            if doc_type == 'regulation':
                chunks = self._chunk_regulation(content, file_name)
            elif doc_type == 'court_decision':
                chunks = self._chunk_court_decision(content, file_name)
            else:
                chunks = self._chunk_generic(content, file_name)
            
            all_chunks.extend(chunks)
        
        return all_chunks
    
    def _identify_document_type(self, file_name: str, content: str) -> str:
        """Doküman tipini belirle"""
        if 'YÖNETMELİĞİ' in content or 'KANUN' in content or 'Regulation' in file_name:
            return 'regulation'
        elif 'MAHKEMESİ' in content or 'KARAR' in content or any(x in file_name for x in ['2025_', '.pdf']):
            return 'court_decision'
        return 'generic'
    
    def _chunk_regulation(self, content: str, file_name: str) -> List[Chunk]:
        """
        Yönetmelik/Kanun chunking stratejisi:
        - Her madde = 1 chunk (maddeler genelde kısa ve bağımsız)
        - Çok uzun maddeler alt-paragraflara bölünür
        - İlgili maddeler gruplandırılabilir (aynı konu)
        """
        chunks = []
        
        # Minimal header: Sadece yönetmelik adı (1 satır)
        header_line = content.split('\n')[0].strip()
        
        # Maddelere böl
        articles = self._split_into_articles(content)
        
        for i, article in enumerate(articles):
            if len(article.strip()) < self.min_chunk_size:
                continue
            
            # Madde numarasını tespit et
            madde_match = re.search(r'MADDE\s+(\d+)', article)
            madde_no = madde_match.group(1) if madde_match else str(i+1)
            
            # Madde başlığını çıkar (ilk satır)
            article_lines = article.strip().split('\n')
            article_title = article_lines[0] if article_lines else f"MADDE {madde_no}"
            
            # Chunk içeriği: Header + Tam madde
            chunk_content = f"{header_line}\n\n{article.strip()}"
            
            # Eğer madde çok uzunsa (>max_chunk_size), alt-paragraflara böl
            if len(chunk_content) > self.max_chunk_size:
                sub_chunks = self._split_long_article(article, header_line, madde_no)
                chunks.extend(sub_chunks)
            else:
                chunk = Chunk(
                    id=f"uuid_{uuid.uuid4()}_{file_name}_madde_{madde_no}",
                    content=chunk_content,
                    metadata={
                        'file_name': file_name,
                        'doc_type': 'regulation',
                        'article_number': madde_no,
                        'article_title': article_title,
                        'char_count': len(chunk_content),
                        'is_complete_article': True
                    },
                    chunk_type='article'
                )
                chunks.append(chunk)
        
        # Ekleri (EK-1, EK-2) ayrı chunk yap
        annex_chunks = self._extract_annexes(content, file_name, header_line)
        chunks.extend(annex_chunks)
        
        return chunks
    
    def _chunk_court_decision(self, content: str, file_name: str) -> List[Chunk]:
        """
        Mahkeme kararı chunking stratejisi:
        - Karar = 3-4 büyük chunk (dava bilgisi, gerekçe, değerlendirme, hüküm)
        - Her chunk bağımsız anlamlı olmalı
        - Çok uzun bölümler mantıksal alt-bölümlere ayrılır
        """
        chunks = []
        
        # Minimal header: Mahkeme + Esas/Karar No + Dava Tipi
        header = self._extract_decision_header(content)
        
        # Ana bölümleri tespit et
        sections = self._identify_decision_sections_smart(content, header)
        
        # Bölümleri anlamlı şekilde grupla ve chunk'la
        for section_name, section_content in sections.items():
            if not section_content or len(section_content.strip()) < self.min_chunk_size:
                continue
            
            # Bölüm yeterince kısa mı? Direkt chunk yap
            if len(section_content) <= self.max_chunk_size:
                chunk = Chunk(
                    id=f"uuid_{uuid.uuid4()}_{file_name}_{section_name}",
                    content=f"{header}\n\n{section_content}",
                    metadata={
                        'file_name': file_name,
                        'doc_type': 'court_decision',
                        'section': section_name,
                        'char_count': len(section_content),
                        'is_complete_section': True
                    },
                    chunk_type='decision_section'
                )
                chunks.append(chunk)
            else:
                # Çok uzunsa: anlamsal sınırlarda böl (paragraf bazlı)
                sub_chunks = self._split_long_section_smart(
                    section_content, header, section_name, file_name
                )
                chunks.extend(sub_chunks)
        
        return chunks
    
    def _identify_decision_sections_smart(self, content: str, header: str) -> Dict[str, str]:
        """
        Mahkeme kararını 3-4 ana bölüme ayır (RAG için optimal)
        
        1. dava_bilgisi: DAVA + GEREĞİ DÜŞÜNÜLDÜ (taraf iddiaları)
        2. gerekce_deliller: DELİLLERİN DEĞERLENDİRİLMESİ (tüm analiz)
        3. hukum_karar: HÜKÜM + KARAR (sonuç)
        """
        sections = {}
        
        # Header'ı temizle
        header_lines = set(header.split('\n'))
        clean_lines = []
        for line in content.split('\n'):
            line = line.strip()
            if line and line not in header_lines:
                clean_lines.append(line)
        
        full_text = '\n'.join(clean_lines)
        
        # 1. DAVA BİLGİSİ: Başlangıçtan "DELİLLERİN" kısmına kadar
        dava_match = re.search(
            r'((?:DAVA:|DAVAcı|GEREĞİ DÜŞÜNÜLDÜ).*?)(?=DELİLLERİN DEĞERLENDİRİLMESİ|GEREKÇE:|$)',
            full_text,
            re.DOTALL | re.IGNORECASE
        )
        if dava_match:
            sections['dava_bilgisi'] = dava_match.group(1).strip()
        
        # 2. GEREKÇE VE DELİLLER: "DELİLLERİN" kısmından "HÜKÜM"e kadar
        gerekce_match = re.search(
            r'(DELİLLERİN DEĞERLENDİRİLMESİ.*?)(?=HÜKÜM:|KARAR:|$)',
            full_text,
            re.DOTALL | re.IGNORECASE
        )
        if gerekce_match:
            sections['gerekce_deliller'] = gerekce_match.group(1).strip()
        
        # 3. HÜKÜM VE KARAR: "HÜKÜM:" kısmından sona kadar
        hukum_match = re.search(
            r'(HÜKÜM:.*)',
            full_text,
            re.DOTALL | re.IGNORECASE
        )
        if hukum_match:
            sections['hukum_karar'] = hukum_match.group(1).strip()
        
        return sections
    
    def _split_long_section_smart(self, text: str, header: str, 
                                   section_name: str, file_name: str) -> List[Chunk]:
        """
        Uzun bölümleri anlamsal sınırlarda böl
        - Paragraflara göre böl (çift newline)
        - Overlap ekle (bağlam kaybını önle)
        - Her chunk'ın başına header ekle
        """
        chunks = []
        paragraphs = re.split(r'\n\s*\n', text)
        
        current_chunk = ""
        chunk_index = 0
        overlap_text = ""
        
        for i, para in enumerate(paragraphs):
            para = para.strip()
            if not para:
                continue
            
            # Chunk boyut kontrolü
            if len(current_chunk) + len(para) + len(overlap_text) <= self.max_chunk_size:
                current_chunk += para + "\n\n"
            else:
                # Chunk'ı kaydet
                if current_chunk.strip():
                    chunk = Chunk(
                        id=f"uuid_{uuid.uuid4()}_{file_name}_{section_name}_part{chunk_index}",
                        content=f"{header}\n\n{current_chunk.strip()}",
                        metadata={
                            'file_name': file_name,
                            'doc_type': 'court_decision',
                            'section': section_name,
                            'part_index': chunk_index,
                            'total_parts': '?',  # Sonra güncellenecek
                            'char_count': len(current_chunk),
                            'is_complete_section': False
                        },
                        chunk_type='decision_section_part'
                    )
                    chunks.append(chunk)
                    chunk_index += 1
                    
                    # Overlap: son paragrafı tut
                    overlap_text = para + "\n\n"
                    current_chunk = overlap_text
                else:
                    current_chunk = para + "\n\n"
        
        # Son chunk
        if current_chunk.strip():
            chunk = Chunk(
                id=f"uuid_{uuid.uuid4()}_{file_name}_{section_name}_part{chunk_index}",
                content=f"{header}\n\n{current_chunk.strip()}",
                metadata={
                    'file_name': file_name,
                    'doc_type': 'court_decision',
                    'section': section_name,
                    'part_index': chunk_index,
                    'total_parts': chunk_index + 1,
                    'char_count': len(current_chunk),
                    'is_complete_section': False
                },
                chunk_type='decision_section_part'
            )
            chunks.append(chunk)
        
        # Total parts güncelle
        total = len(chunks)
        for chunk in chunks:
            chunk.metadata['total_parts'] = total
        
        return chunks
    
    def _split_long_article(self, article: str, header: str, madde_no: str) -> List[Chunk]:
        """Uzun maddeyi alt-paragraflara böl"""
        chunks = []
        
        # Madde başlığı
        article_lines = article.strip().split('\n')
        article_title = article_lines[0] if article_lines else f"MADDE {madde_no}"
        
        # Fıkralara böl (a), b), c) veya (1), (2), (3) formatı)
        paragraphs = re.split(r'\n(?=[a-z]\)|(?:\(\d+\)))', article)
        
        current_chunk = article_title + "\n\n"
        chunk_index = 0
        
        for para in paragraphs[1:] if len(paragraphs) > 1 else paragraphs:
            para = para.strip()
            if not para:
                continue
            
            if len(current_chunk) + len(para) <= self.max_chunk_size:
                current_chunk += para + "\n"
            else:
                # Chunk kaydet
                if current_chunk.strip():
                    chunk = Chunk(
                        id=f"uuid_{uuid.uuid4()}_madde_{madde_no}_part{chunk_index}",
                        content=f"{header}\n\n{current_chunk.strip()}",
                        metadata={
                            'doc_type': 'regulation',
                            'article_number': madde_no,
                            'part_index': chunk_index,
                            'char_count': len(current_chunk),
                            'is_complete_article': False
                        },
                        chunk_type='article_part'
                    )
                    chunks.append(chunk)
                    chunk_index += 1
                
                current_chunk = article_title + "\n\n" + para + "\n"
        
        # Son chunk
        if current_chunk.strip():
            chunk = Chunk(
                id=f"uuid_{uuid.uuid4()}_madde_{madde_no}_part{chunk_index}",
                content=f"{header}\n\n{current_chunk.strip()}",
                metadata={
                    'doc_type': 'regulation',
                    'article_number': madde_no,
                    'part_index': chunk_index,
                    'char_count': len(current_chunk),
                    'is_complete_article': False
                },
                chunk_type='article_part'
            )
            chunks.append(chunk)
        
        return chunks
    
    def _chunk_generic(self, content: str, file_name: str) -> List[Chunk]:
        """Genel dokümanlar için paragraf bazlı chunking"""
        chunks = []
        
        # Paragraflara böl
        paragraphs = re.split(r'\n\s*\n', content)
        
        current_chunk = ""
        chunk_count = 0
        overlap_para = ""
        
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            
            if len(current_chunk) + len(para) + len(overlap_para) < self.max_chunk_size:
                current_chunk += para + "\n\n"
            else:
                if len(current_chunk.strip()) >= self.min_chunk_size:
                    chunk = Chunk(
                        id=f"uuid_{uuid.uuid4()}_{file_name}_chunk_{chunk_count}",
                        content=current_chunk.strip(),
                        metadata={
                            'file_name': file_name,
                            'doc_type': 'generic',
                            'chunk_number': chunk_count,
                            'char_count': len(current_chunk)
                        },
                        chunk_type='generic'
                    )
                    chunks.append(chunk)
                    chunk_count += 1
                
                # Overlap
                overlap_para = para + "\n\n"
                current_chunk = overlap_para
        
        # Son chunk
        if len(current_chunk.strip()) >= self.min_chunk_size:
            chunk = Chunk(
                id=f"uuid_{uuid.uuid4()}_{file_name}_chunk_{chunk_count}",
                content=current_chunk.strip(),
                metadata={
                    'file_name': file_name,
                    'doc_type': 'generic',
                    'chunk_number': chunk_count,
                    'char_count': len(current_chunk)
                },
                chunk_type='generic'
            )
            chunks.append(chunk)
        
        return chunks
    
    def _extract_decision_header(self, content: str) -> str:
        """Mahkeme kararı header'ını minimal tut"""
        lines = content.split('\n')
        header_lines = []
        
        for line in lines[:20]:
            line = line.strip()
            if any(kw in line for kw in ['MAHKEMESİ', 'ESAS NO', 'KARAR NO', 'DAVA:']):
                header_lines.append(line)
                if 'DAVA:' in line:
                    break
        
        return '\n'.join(header_lines[:4]) if header_lines else lines[0]
    
    def _split_into_articles(self, content: str) -> List[str]:
        """Maddelere böl"""
        pattern = r'(MADDE\s+\d+.*?)(?=MADDE\s+\d+|EK-\d+|$)'
        articles = re.findall(pattern, content, re.DOTALL)
        return [art.strip() for art in articles if len(art.strip()) > 50]
    
    def _extract_annexes(self, content: str, file_name: str, header: str) -> List[Chunk]:
        """Ekleri çıkar"""
        chunks = []
        annex_pattern = r'(EK-\d+.*?)(?=EK-\d+|$)'
        annexes = re.findall(annex_pattern, content, re.DOTALL)
        
        for i, annex in enumerate(annexes):
            if len(annex.strip()) >= self.min_chunk_size:
                chunk = Chunk(
                    id=f"uuid_{uuid.uuid4()}_{file_name}_ek_{i+1}",
                    content=f"{header}\n\n{annex.strip()}",
                    metadata={
                        'file_name': file_name,
                        'doc_type': 'regulation',
                        'section': 'annex',
                        'annex_number': i+1,
                        'char_count': len(annex)
                    },
                    chunk_type='annex'
                )
                chunks.append(chunk)
        
        return chunks
    
    def save_chunks(self, chunks: List[Chunk], output_file: str):
        """Chunk'ları JSON olarak kaydet"""
        chunks_data = [
            {
                'id': chunk.id,
                'content': chunk.content,
                'metadata': chunk.metadata,
                'chunk_type': chunk.chunk_type
            }
            for chunk in chunks
        ]
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(chunks_data, f, ensure_ascii=False, indent=2)
        
        print(f"✓ {len(chunks)} chunk kaydedildi: {output_file}")
    
    def print_statistics(self, chunks: List[Chunk]):
        """Detaylı istatistikler"""
        print("\n" + "="*60)
        print("RAG-OPTIMIZED CHUNKING İSTATİSTİKLERİ")
        print("="*60)
        print(f"Toplam chunk sayısı: {len(chunks)}")
        
        # Tip dağılımı
        type_counts = {}
        for chunk in chunks:
            doc_type = chunk.metadata.get('doc_type', 'unknown')
            type_counts[doc_type] = type_counts.get(doc_type, 0) + 1
        
        print("\n📊 Doküman Tiplerine Göre:")
        for doc_type, count in sorted(type_counts.items()):
            print(f"   {doc_type}: {count} chunk")
        
        # Mahkeme kararı bölümleri
        section_counts = {}
        complete_sections = 0
        for chunk in chunks:
            if chunk.metadata.get('doc_type') == 'court_decision':
                section = chunk.metadata.get('section', 'unknown')
                section_counts[section] = section_counts.get(section, 0) + 1
                if chunk.metadata.get('is_complete_section'):
                    complete_sections += 1
        
        if section_counts:
            print("\n⚖️  Mahkeme Kararı Bölümleri:")
            for section, count in sorted(section_counts.items()):
                print(f"   {section}: {count} chunk")
            print(f"   Tam bölüm chunk'ları: {complete_sections}")
        
        # Boyut analizi
        sizes = [chunk.metadata.get('char_count', 0) for chunk in chunks]
        print(f"\n📏 Chunk Boyutları (karakter):")
        print(f"   Ortalama: {sum(sizes) / len(sizes):.0f}")
        print(f"   Minimum: {min(sizes)}")
        print(f"   Maximum: {max(sizes)}")
        print(f"   Medyan: {sorted(sizes)[len(sizes)//2]}")
        
        # Kalite metrikleri
        complete_articles = sum(1 for c in chunks if c.metadata.get('is_complete_article'))
        complete_sections = sum(1 for c in chunks if c.metadata.get('is_complete_section'))
        
        print(f"\n✅ Anlamsal Bütünlük:")
        print(f"   Tam madde chunk'ları: {complete_articles}")
        print(f"   Tam bölüm chunk'ları: {complete_sections}")
        print(f"   Parçalanmış chunk'lar: {len(chunks) - complete_articles - complete_sections}")


# KULLANIM ÖRNEĞİ
if __name__ == "__main__":
    # RAG için optimize edilmiş parametreler
    chunker = LegalDocumentChunker(
        max_chunk_size=3000,    # Daha büyük: anlamsal bütünlük
        overlap_size=300,       # Daha fazla: bağlam korunumu
        min_chunk_size=100      # Daha yüksek: çok küçük chunk'ları önle
    )
    
    # Dokümanları chunk'la
    print("Dokümanlar chunk'lanıyor...")
    chunks = chunker.chunk_documents('/Users/beyzaasan/Projects/HukukPusulasi/rag_files/extracted_pdf_texts_RAG_format.json')
    
    # İstatistikleri göster
    chunker.print_statistics(chunks)
    
    # Chunk'ları kaydet
    chunker.save_chunks(chunks, '/Users/beyzaasan/Projects/HukukPusulasi/rag_files/chunked_documents.json')
    
    # İlk birkaç chunk'ı örnek olarak göster
    print("\n=== ÖRNEK CHUNK'LAR ===")
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"ID: {chunk.id}")
        print(f"Tip: {chunk.chunk_type}")
        print(f"Metadata: {chunk.metadata}")
        print(f"İçerik (ilk 200 karakter):\n{chunk.content[:200]}...")

## Tokenize chunks

In [ ]:
import json
import re
from typing import List, Dict, Tuple
from dataclasses import dataclass
from transformers import AutoTokenizer
import numpy as np

@dataclass
class TokenizedChunk:
    """Tokenize edilmiş chunk veri yapısı"""
    id: str
    original_content: str
    tokens: List[int]
    token_count: int
    metadata: Dict
    chunk_type: str
    embeddings_ready: bool = False


class LegalDocumentTokenizer:
    """
    Türkçe hukuki dokümanlar için tokenization sistemi
    Model: emrecan/bert-base-turkish-cased-v1
    
    Temel Prensipler:
    1. Tokenizasyon: BERT tokenizer ile
    2. Token limitleri: Max 512 token (BERT standarı)
    3. Overflow handling: Uzun metinler bölünür
    4. Special tokens: CLS, SEP, PAD düzgün kullanılır
    """
    
    def __init__(self, 
                 model_name: str = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr",
                 max_tokens: int = 512,
                 padding_strategy: str = "max_length"):
        """
        Args:
            model_name: Hugging Face model ismi
            max_tokens: Maksimum token sayısı (BERT için 512)
            padding_strategy: 'max_length' veya 'do_not_pad'
        """
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.padding_strategy = padding_strategy
        
        print(f"Tokenizer yükleniyor: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        print(f"✓ Tokenizer yüklendi")
        print(f"   Vocab size: {len(self.tokenizer)}")
        print(f"   Model max length: {self.tokenizer.model_max_length}")
    
    def tokenize_chunks(self, 
                       json_file_path: str,
                       handle_overflow: str = "split") -> List[TokenizedChunk]:
        """
        JSON dosyasındaki chunk'ları tokenize et
        
        Args:
            json_file_path: Chunk'ları içeren JSON dosyası
            handle_overflow: 'split' (uzun metni böl) veya 'truncate' (kes)
        
        Returns:
            Tokenize edilmiş chunk'ların listesi
        """
        with open(json_file_path, 'r', encoding='utf-8') as f:
            chunks_data = json.load(f)
        
        tokenized_chunks = []
        overflow_count = 0
        
        for i, chunk_data in enumerate(chunks_data):
            if (i + 1) % 50 == 0:
                print(f"   {i + 1}/{len(chunks_data)} chunk tokenize edildi...")
            
            # Chunk'ı tokenize et
            result = self._tokenize_single_chunk(
                chunk_data,
                handle_overflow=handle_overflow
            )
            
            if isinstance(result, list):
                # Overflow durumu: birden fazla chunk oluştu
                tokenized_chunks.extend(result)
                overflow_count += len(result) - 1
            else:
                tokenized_chunks.append(result)
        
        print(f"✓ {len(chunks_data)} chunk -> {len(tokenized_chunks)} tokenized chunk")
        print(f"   Overflow chunks oluşturuldu: {overflow_count}")
        
        return tokenized_chunks
    
    def _tokenize_single_chunk(self, 
                              chunk_data: Dict,
                              handle_overflow: str = "split") -> Tuple[TokenizedChunk] | TokenizedChunk:
        """
        Tekil bir chunk'ı tokenize et
        
        Returns:
            - Eğer overflow yok: TokenizedChunk
            - Eğer overflow var: TokenizedChunk listesi
        """
        chunk_id = chunk_data['id']
        content = chunk_data['content']
        metadata = chunk_data['metadata']
        chunk_type = chunk_data['chunk_type']
        
        # Tokenize et
        encoding = self.tokenizer(
            content,
            max_length=self.max_tokens,
            truncation=False,  # İlk olarak truncation yapma
            padding=False,     # Başta padding yapma
            return_tensors=None,
            return_token_type_ids=True,
            return_attention_mask=True
        )
        
        tokens = encoding['input_ids']
        token_count = len(tokens)
        
        # Token sayısı kontrol et
        if token_count <= self.max_tokens:
            # Normal case: tokenlar sınırda
            tokenized_chunk = TokenizedChunk(
                id=chunk_id,
                original_content=content,
                tokens=tokens,
                token_count=token_count,
                metadata=metadata,
                chunk_type=chunk_type,
                embeddings_ready=True
            )
            return tokenized_chunk
        
        else:
            # Overflow case: chunk'ı böl veya kes
            if handle_overflow == "split":
                return self._split_overflow_chunk(
                    chunk_id, content, metadata, chunk_type
                )
            else:  # truncate
                tokens = tokens[:self.max_tokens]
                tokenized_chunk = TokenizedChunk(
                    id=f"{chunk_id}_truncated",
                    original_content=content,
                    tokens=tokens,
                    token_count=len(tokens),
                    metadata={**metadata, 'truncated': True},
                    chunk_type=chunk_type,
                    embeddings_ready=True
                )
                return tokenized_chunk
    
    def _split_overflow_chunk(self, 
                             chunk_id: str,
                             content: str,
                             metadata: Dict,
                             chunk_type: str,
                             overlap_ratio: float = 0.1) -> List[TokenizedChunk]:
        """
        Overflow chunk'ı overlap ile böl
        
        Args:
            overlap_ratio: Bölümlü chunk'lar arasında overlap oranı (0.1 = %10)
        """
        result_chunks = []
        
        # Paragraf bazında böl
        paragraphs = re.split(r'\n\s*\n', content)
        
        current_text = ""
        part_index = 0
        overlap_text = ""
        
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            
            # Test et: Bu paragraf eklenebilir mi?
            test_text = overlap_text + current_text + para
            test_encoding = self.tokenizer(
                test_text,
                truncation=False,
                padding=False,
                return_tensors=None
            )
            
            if len(test_encoding['input_ids']) <= self.max_tokens:
                # Ekle
                current_text += para + "\n\n"
            else:
                # Chunk kaydet
                if current_text.strip():
                    chunk_text = overlap_text + current_text
                    tokens = self.tokenizer(
                        chunk_text,
                        max_length=self.max_tokens,
                        truncation=True,
                        padding=False,
                        return_tensors=None
                    )['input_ids']
                    
                    tokenized_chunk = TokenizedChunk(
                        id=f"{chunk_id}_part{part_index}",
                        original_content=chunk_text,
                        tokens=tokens,
                        token_count=len(tokens),
                        metadata={
                            **metadata,
                            'overflow_handling': 'split',
                            'part_index': part_index,
                            'is_overflow_part': True
                        },
                        chunk_type=chunk_type,
                        embeddings_ready=True
                    )
                    result_chunks.append(tokenized_chunk)
                    part_index += 1
                
                # Overlap kısm belirle
                overlap_sentences = self._get_overlap_text(
                    current_text, overlap_ratio
                )
                overlap_text = overlap_sentences
                current_text = para + "\n\n"
        
        # Son chunk
        if current_text.strip():
            chunk_text = overlap_text + current_text
            tokens = self.tokenizer(
                chunk_text,
                max_length=self.max_tokens,
                truncation=True,
                padding=False,
                return_tensors=None
            )['input_ids']
            
            tokenized_chunk = TokenizedChunk(
                id=f"{chunk_id}_part{part_index}",
                original_content=chunk_text,
                tokens=tokens,
                token_count=len(tokens),
                metadata={
                    **metadata,
                    'overflow_handling': 'split',
                    'part_index': part_index,
                    'is_overflow_part': True
                },
                chunk_type=chunk_type,
                embeddings_ready=True
            )
            result_chunks.append(tokenized_chunk)
        
        return result_chunks if result_chunks else [
            TokenizedChunk(
                id=f"{chunk_id}_truncated",
                original_content=content,
                tokens=self.tokenizer(content, max_length=self.max_tokens, truncation=True)['input_ids'],
                token_count=self.max_tokens,
                metadata={**metadata, 'truncated': True},
                chunk_type=chunk_type,
                embeddings_ready=True
            )
        ]
    
    def _get_overlap_text(self, text: str, ratio: float) -> str:
        """
        Metin sonundan belirli oranında overlap kısım al
        """
        sentences = re.split(r'(?<=[.!?])\s+', text)
        overlap_count = max(1, int(len(sentences) * ratio))
        overlap_text = ' '.join(sentences[-overlap_count:])
        return overlap_text
    
    def add_special_tokens_info(self, tokenized_chunks: List[TokenizedChunk]) -> List[TokenizedChunk]:
        """
        Her chunk'a special token bilgisi ekle
        (Embedding modeline geçmeden önce)
        """
        for chunk in tokenized_chunks:
            # Token'ları dekod et (debug için)
            token_texts = self.tokenizer.convert_ids_to_tokens(chunk.tokens)
            
            # Special token'ları sayıştır
            special_token_count = sum(
                1 for t in token_texts 
                if t in ['[CLS]', '[SEP]', '[PAD]', '[UNK]']
            )
            
            chunk.metadata['token_info'] = {
                'total_tokens': chunk.token_count,
                'special_tokens': special_token_count,
                'content_tokens': chunk.token_count - special_token_count,
                'token_sample': token_texts[:10]  # İlk 10 token örneği
            }
        
        return tokenized_chunks
    
    def save_tokenized_chunks(self, 
                             tokenized_chunks: List[TokenizedChunk],
                             output_file: str):
        """
        Tokenize edilmiş chunk'ları JSON olarak kaydet
        """
        chunks_data = [
            {
                'id': chunk.id,
                'original_content': chunk.original_content,
                'tokens': chunk.tokens,
                'token_count': chunk.token_count,
                'metadata': chunk.metadata,
                'chunk_type': chunk.chunk_type,
                'embeddings_ready': chunk.embeddings_ready
            }
            for chunk in tokenized_chunks
        ]
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(chunks_data, f, ensure_ascii=False, indent=2)
        
        print(f"✓ {len(tokenized_chunks)} tokenized chunk kaydedildi: {output_file}")
    
    def print_tokenization_statistics(self, tokenized_chunks: List[TokenizedChunk]):
        """
        Tokenizasyon istatistikleri
        """
        print("\n" + "="*60)
        print("TOKENIZASYON İSTATİSTİKLERİ")
        print("="*60)
        print(f"Toplam tokenized chunk: {len(tokenized_chunks)}")
        print(f"Model: {self.model_name}")
        print(f"Max token length: {self.max_tokens}")
        
        # Token dağılımı
        token_counts = [chunk.token_count for chunk in tokenized_chunks]
        print(f"\n📊 Token Sayı Dağılımı:")
        print(f"   Ortalama: {np.mean(token_counts):.1f}")
        print(f"   Minimum: {min(token_counts)}")
        print(f"   Maximum: {max(token_counts)}")
        print(f"   Medyan: {np.median(token_counts):.1f}")
        print(f"   Std Dev: {np.std(token_counts):.1f}")
        
        # Overflow işlenmiş chunk'lar
        overflow_chunks = sum(
            1 for c in tokenized_chunks 
            if c.metadata.get('is_overflow_part')
        )
        truncated_chunks = sum(
            1 for c in tokenized_chunks 
            if c.metadata.get('truncated')
        )
        
        print(f"\n⚠️  Overflow İşleme:")
        print(f"   Bölünmüş chunk parçaları: {overflow_chunks}")
        print(f"   Truncate edilmiş chunk'lar: {truncated_chunks}")
        print(f"   Normal chunk'lar: {len(tokenized_chunks) - overflow_chunks - truncated_chunks}")
        
        # Chunk tipi dağılımı
        type_counts = {}
        for chunk in tokenized_chunks:
            chunk_type = chunk.chunk_type
            type_counts[chunk_type] = type_counts.get(chunk_type, 0) + 1
        
        print(f"\n📋 Chunk Tiplerine Göre:")
        for chunk_type, count in sorted(type_counts.items()):
            avg_tokens = np.mean([
                c.token_count for c in tokenized_chunks 
                if c.chunk_type == chunk_type
            ])
            print(f"   {chunk_type}: {count} chunk (ort. {avg_tokens:.0f} token)")
        
        # Embedding hazırlığı
        ready_count = sum(
            1 for c in tokenized_chunks 
            if c.embeddings_ready
        )
        print(f"\n✅ Embedding Hazırlığı:")
        print(f"   Embedding'e hazır: {ready_count}")
        print(f"   Toplam: {len(tokenized_chunks)}")


# KULLANIM ÖRNEĞİ
if __name__ == "__main__":
    # Tokenizer oluştur
    
    tokenizer = LegalDocumentTokenizer(
        model_name="emrecan/bert-base-turkish-cased-mean-nli-stsb-tr",
        max_tokens=512,
        padding_strategy="max_length"
    )
    
    # Chunk'ları tokenize et
    print("\nChunk'lar tokenize ediliyor...")
    tokenized_chunks = tokenizer.tokenize_chunks(
        json_file_path='/Users/beyzaasan/Projects/HukukPusulasi/rag_files/chunked_documents.json',
        handle_overflow='split'
    )
    
    # Special token bilgisi ekle
    print("\nSpecial token bilgisi ekleniyor...")
    tokenized_chunks = tokenizer.add_special_tokens_info(tokenized_chunks)
    
    # İstatistikleri göster
    tokenizer.print_tokenization_statistics(tokenized_chunks)
    
    # Tokenized chunk'ları kaydet
    tokenizer.save_tokenized_chunks(
        tokenized_chunks,
        '/Users/beyzaasan/Projects/HukukPusulasi/rag_files/tokenized_chunks.json'
    )
    
    # Örnek tokenized chunk
    print("\n=== ÖRNEK TOKENIZED CHUNK ===")
    if tokenized_chunks:
        chunk = tokenized_chunks[4]
        print(f"\nID: {chunk.id}")
        print(f"Token sayısı: {chunk.token_count}")
        print(f"Token dizisi (ilk 20): {chunk.tokens[:20]}")
        print(f"Metadata: {chunk.metadata}")
        print(f"İçerik (ilk 150 karakter): {chunk.original_content[:150]}...")
        
        # Token'ları görüntüle
        token_texts = tokenizer.tokenizer.convert_ids_to_tokens(chunk.tokens[:20])
        print(f"Token metinleri (ilk 20): {token_texts}")

## Vector Embedding

In [ ]:
import json
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from typing import List, Dict, Optional
import numpy as np
from tqdm import tqdm
import os
from datetime import datetime
import torch

class LegalVectorStore:
    """
    ChromaDB ile hukuki dokümanlar için vektör store
    
    Temel Prensipler:
    1. Türkçe BERT embedding: emrecan/bert-base-turkish-cased-mean-nli-stsb-tr
    2. Metadata filtering: Doküman tipi, madde no, bölüm vb.
    3. Similarity search: Cosine similarity
    4. Persistent storage: Versiyon kontrolü ile
    """
    
    def __init__(self, 
                 persist_directory: str = "./chroma_db",
                 collection_name: str = "legal_documents",
                 model_name: str = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"):
        """
        Args:
            persist_directory: ChromaDB'nin kaydedileceği dizin
            collection_name: Collection ismi
            model_name: Hugging Face embedding modeli
        """
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.model_name = model_name
        
        # Dizin oluştur
        os.makedirs(persist_directory, exist_ok=True)
        
        print(f"ChromaDB başlatılıyor...")
        print(f"   Persist directory: {persist_directory}")
        print(f"   Collection: {collection_name}")
        print(f"   Model: {model_name}")
        
        # ChromaDB client oluştur (persistent)
        self.client = chromadb.PersistentClient(
            path=persist_directory,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )
        
        if torch.cuda.is_available():
          device = "cuda"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
          device = "mps"
        else:
          device = "cpu"
        
        # Türkçe BERT embedding function
        print(f"\nEmbedding modeli yükleniyor (Device: {device})...")
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name,
            device=device
        )
        print(f"✓ Embedding modeli yüklendi")
        
        # Collection oluştur veya yükle
        try:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
            print(f"✓ Mevcut collection yüklendi: {self.collection.count()} doküman")
        except Exception:
            self.collection = self.client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_function,
                metadata={
                    "description": "Turkish legal documents RAG system",
                    "model": model_name,
                    "created_at": datetime.now().isoformat()
                }
            )
            print(f"✓ Yeni collection oluşturuldu")
    
    def add_tokenized_chunks(self, 
                            json_file_path: str,
                            batch_size: int = 100,
                            skip_existing: bool = True) -> Dict:
        """
        Tokenize edilmiş chunk'ları ChromaDB'ye ekle
        
        Args:
            json_file_path: Tokenized chunks JSON dosyası
            batch_size: Her batch'te kaç chunk ekleneceği
            skip_existing: Mevcut ID'leri atla
        
        Returns:
            İstatistik dict'i
        """
        print(f"\nTokenized chunks yükleniyor: {json_file_path}")
        with open(json_file_path, 'r', encoding='utf-8') as f:
            chunks_data = json.load(f)
        
        print(f"✓ {len(chunks_data)} chunk yüklendi")
        
        # Mevcut ID'leri kontrol et
        existing_ids = set()
        if skip_existing:
            try:
                existing_items = self.collection.get()
                existing_ids = set(existing_items['ids'])
                print(f"   Mevcut chunk sayısı: {len(existing_ids)}")
            except Exception:
                pass
        
        # Eklenecek chunk'ları filtrele
        new_chunks = [
            chunk for chunk in chunks_data 
            if chunk['id'] not in existing_ids
        ]
        
        if not new_chunks:
            print("✓ Tüm chunk'lar zaten mevcut, ekleme yapılmadı")
            return {
                'total': len(chunks_data),
                'existing': len(existing_ids),
                'added': 0,
                'skipped': len(chunks_data)
            }
        
        print(f"   Eklenecek yeni chunk: {len(new_chunks)}")
        
        # Batch halinde ekle
        stats = {
            'total': len(chunks_data),
            'existing': len(existing_ids),
            'added': 0,
            'failed': 0,
            'batches': 0
        }
        
        print(f"\nChunk'lar ChromaDB'ye ekleniyor (batch_size={batch_size})...")
        
        for i in tqdm(range(0, len(new_chunks), batch_size), desc="Embedding & Adding"):
            batch = new_chunks[i:i+batch_size]
            
            try:
                # Batch verisini hazırla
                ids = [chunk['id'] for chunk in batch]
                documents = [chunk['original_content'] for chunk in batch]
                metadatas = [self._prepare_metadata(chunk) for chunk in batch]
                
                # ChromaDB'ye ekle (embedding otomatik yapılır)
                self.collection.add(
                    ids=ids,
                    documents=documents,
                    metadatas=metadatas
                    )
                
                stats['added'] += len(batch)
                stats['batches'] += 1
                
            except Exception as e:
                print(f"\n⚠️  Batch {i//batch_size + 1} hata: {str(e)}")
                stats['failed'] += len(batch)
        
        print(f"\n✓ Ekleme tamamlandı!")
        self._print_add_statistics(stats)
        
        return stats
    
    def _prepare_metadata(self, chunk_data: Dict) -> Dict:
        """
        ChromaDB metadata hazırla
        Not: ChromaDB metadata'da nested dict/list desteklemiyor,
        sadece str, int, float, bool
        """
        metadata = chunk_data['metadata'].copy()
        
        # Temel alanlar
        result = {
            'file_name': str(metadata.get('file_name', '')),
            'doc_type': str(metadata.get('doc_type', '')),
            'chunk_type': str(chunk_data['chunk_type']),
            'token_count': int(chunk_data['token_count']),
            'char_count': int(metadata.get('char_count', 0)),
        }
        
        # Tip-spesifik alanlar
        if metadata.get('doc_type') == 'regulation':
            if 'article_number' in metadata:
                result['article_number'] = str(metadata['article_number'])
            if 'article_title' in metadata:
                result['article_title'] = str(metadata['article_title'])
            if 'is_complete_article' in metadata:
                result['is_complete_article'] = bool(metadata['is_complete_article'])
        
        elif metadata.get('doc_type') == 'court_decision':
            if 'section' in metadata:
                result['section'] = str(metadata['section'])
            if 'is_complete_section' in metadata:
                result['is_complete_section'] = bool(metadata['is_complete_section'])
        
        # Overflow bilgisi
        if metadata.get('is_overflow_part'):
            result['is_overflow_part'] = True
            result['part_index'] = int(metadata.get('part_index', 0))
        
        if metadata.get('truncated'):
            result['truncated'] = True
        
        return result
    
    def search_similar(self, 
                      query: str,
                      n_results: int = 5,
                      where: Optional[Dict] = None,
                      where_document: Optional[Dict] = None) -> Dict:
        """
        Benzer chunk'ları bul
        
        Args:
            query: Arama sorgusu (Türkçe)
            n_results: Kaç sonuç döndürülecek
            where: Metadata filtreleri (örn: {"doc_type": "regulation"})
            where_document: Doküman içerik filtreleri
        
        Returns:
            Sonuçlar dict'i
        """
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where,
            where_document=where_document,
            include=["documents", "metadatas", "distances"]
        )
        
        return {
            'query': query,
            'n_results': len(results['ids'][0]),
            'ids': results['ids'][0],
            'documents': results['documents'][0],
            'metadatas': results['metadatas'][0],
            'distances': results['distances'][0],  # Cosine distance (düşük = benzer)
            'similarities': [1 - d for d in results['distances'][0]]  # Similarity skoru
        }
    
    def search_by_article(self, 
                         article_number: str,
                         n_results: int = 3) -> Dict:
        """Belirli bir madde numarasına göre ara"""
        return self.search_similar(
            query=f"Madde {article_number}",
            n_results=n_results,
            where={"article_number": article_number}
        )
    
    def search_by_doc_type(self, 
                          query: str,
                          doc_type: str,
                          n_results: int = 5) -> Dict:
        """Belirli doküman tipinde ara"""
        return self.search_similar(
            query=query,
            n_results=n_results,
            where={"doc_type": doc_type}
        )
    
    def search_regulations(self, query: str, n_results: int = 5) -> Dict:
        """Sadece yönetmeliklerde ara"""
        return self.search_by_doc_type(query, "regulation", n_results)
    
    def search_court_decisions(self, query: str, n_results: int = 5) -> Dict:
        """Sadece mahkeme kararlarında ara"""
        return self.search_by_doc_type(query, "court_decision", n_results)
    
    def get_chunk_by_id(self, chunk_id: str) -> Optional[Dict]:
        """ID'ye göre chunk getir"""
        try:
            result = self.collection.get(
                ids=[chunk_id],
                include=["documents", "metadatas"]
            )
            if result['ids']:
                return {
                    'id': result['ids'][0],
                    'document': result['documents'][0],
                    'metadata': result['metadatas'][0]
                }
        except Exception as e:
            print(f"Chunk bulunamadı: {e}")
        return None
    
    def get_collection_stats(self) -> Dict:
        """Collection istatistikleri"""
        count = self.collection.count()
        
        if count == 0:
            return {
                'total_chunks': 0,
                'doc_types': {},
                'chunk_types': {},
                'avg_token_count': 0
            }
        
        # Tüm metadata'yı çek (örnek için ilk 1000)
        sample_size = min(1000, count)
        sample = self.collection.get(
            limit=sample_size,
            include=["metadatas"]
        )
        
        # İstatistik topla
        doc_types = {}
        chunk_types = {}
        token_counts = []
        
        for meta in sample['metadatas']:
            doc_type = meta.get('doc_type', 'unknown')
            chunk_type = meta.get('chunk_type', 'unknown')
            token_count = meta.get('token_count', 0)
            
            doc_types[doc_type] = doc_types.get(doc_type, 0) + 1
            chunk_types[chunk_type] = chunk_types.get(chunk_type, 0) + 1
            token_counts.append(token_count)
        
        return {
            'total_chunks': count,
            'sampled_chunks': sample_size,
            'doc_types': doc_types,
            'chunk_types': chunk_types,
            'avg_token_count': np.mean(token_counts) if token_counts else 0,
            'model': self.model_name,
            'collection_name': self.collection_name
        }
    
    def print_search_results(self, results: Dict, max_content_length: int = 200):
        """Arama sonuçlarını güzel formatlayarak yazdır"""
        print("\n" + "="*70)
        print(f"ARAMA SONUÇLARI: '{results['query']}'")
        print("="*70)
        print(f"Toplam sonuç: {results['n_results']}\n")
        
        for i, (doc_id, doc, meta, dist, sim) in enumerate(zip(
            results['ids'],
            results['documents'],
            results['metadatas'],
            results['distances'],
            results['similarities']
        ), 1):
            print(f"--- Sonuç {i} ---")
            print(f"ID: {doc_id}")
            print(f"Similarity: {sim:.4f} (distance: {dist:.4f})")
            print(f"Tip: {meta.get('doc_type', 'N/A')} - {meta.get('chunk_type', 'N/A')}")
            
            if meta.get('article_number'):
                print(f"Madde: {meta['article_number']}")
            if meta.get('section'):
                print(f"Bölüm: {meta['section']}")
            
            print(f"Token sayısı: {meta.get('token_count', 'N/A')}")
            print(f"\nİçerik (ilk {max_content_length} karakter):")
            print(doc[:max_content_length] + "..." if len(doc) > max_content_length else doc)
            print()
    
    def _print_add_statistics(self, stats: Dict):
        """Ekleme istatistiklerini yazdır"""
        print("\n" + "="*60)
        print("CHROMADB EKLEME İSTATİSTİKLERİ")
        print("="*60)
        print(f"Toplam chunk: {stats['total']}")
        print(f"Mevcut chunk: {stats['existing']}")
        print(f"Yeni eklenen: {stats['added']}")
        print(f"Başarısız: {stats['failed']}")
        print(f"Batch sayısı: {stats['batches']}")
        print(f"Collection toplam: {self.collection.count()}")
    
    def print_collection_stats(self):
        """Collection istatistiklerini yazdır"""
        stats = self.get_collection_stats()
        
        print("\n" + "="*60)
        print("CHROMADB COLLECTION İSTATİSTİKLERİ")
        print("="*60)
        print(f"Collection: {stats['collection_name']}")
        print(f"Model: {stats['model']}")
        print(f"Toplam chunk: {stats['total_chunks']}")
        
        if stats['total_chunks'] > 0:
            print(f"\n📊 Doküman Tipleri:")
            for doc_type, count in sorted(stats['doc_types'].items()):
                percentage = (count / stats['sampled_chunks']) * 100
                print(f"   {doc_type}: {count} ({percentage:.1f}%)")
            
            print(f"\n📋 Chunk Tipleri:")
            for chunk_type, count in sorted(stats['chunk_types'].items()):
                percentage = (count / stats['sampled_chunks']) * 100
                print(f"   {chunk_type}: {count} ({percentage:.1f}%)")
            
            print(f"\n📏 Token İstatistikleri:")
            print(f"   Ortalama token sayısı: {stats['avg_token_count']:.1f}")
    
    def reset_collection(self):
        """Collection'ı tamamen sil ve yeniden oluştur"""
        print(f"⚠️  Collection siliniyor: {self.collection_name}")
        self.client.delete_collection(name=self.collection_name)
        
        self.collection = self.client.create_collection(
            name=self.collection_name,
            embedding_function=self.embedding_function,
            metadata={
                "description": "Turkish legal documents RAG system",
                "model": self.model_name,
                "created_at": datetime.now().isoformat()
            }
        )
        print(f"✓ Collection yeniden oluşturuldu")


# KULLANIM ÖRNEĞİ
if __name__ == "__main__":
    # Vector store oluştur
    vector_store = LegalVectorStore(
        persist_directory="/Users/beyzaasan/Projects/HukukPusulasi/legal_chroma_db",
        collection_name="legal_documents_v2",
        model_name="emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"
    )
    
    # Tokenized chunk'ları ekle
    print("\n" + "="*70)
    print("CHUNK'LARI CHROMADB'YE EKLEME")
    print("="*70)
    
    stats = vector_store.add_tokenized_chunks(
        json_file_path='/Users/beyzaasan/Projects/HukukPusulasi/rag_files/tokenized_chunks.json',
        batch_size=10,
        skip_existing=True
    )
    
    # Collection istatistikleri
    vector_store.print_collection_stats()
    
    # ÖRNEK ARAMALAR
    print("\n" + "="*70)
    print("ÖRNEK SEMANTIC SEARCH SORULARI")
    print("="*70)
    
    # 1. Genel arama
    print("\n1️⃣  İş kazası tazminatı ile ilgili arama:")
    results = vector_store.search_similar(
        query="İş kazasında işverenin tazminat sorumluluğu nedir?",
        n_results=3
    )
    vector_store.print_search_results(results, max_content_length=300)
    
    # 2. Sadece yönetmeliklerde ara
    print("\n2️⃣  Yönetmeliklerde kişisel verilerin korunması:")
    results = vector_store.search_regulations(
        query="Kişisel verilerin korunması ve işlenmesi kuralları",
        n_results=3
    )
    vector_store.print_search_results(results, max_content_length=300)
    
    # 3. Sadece mahkeme kararlarında ara
    print("\n3️⃣  Mahkeme kararlarında tazminat hesaplama:")
    results = vector_store.search_court_decisions(
        query="Tazminat miktarının hesaplanması ve takdir",
        n_results=3
    )
    vector_store.print_search_results(results, max_content_length=300)
    
    # 4. Specific metadata filter
    print("\n4️⃣  Belirli bir bölümde arama (dava_bilgisi):")
    results = vector_store.search_similar(
        query="Davacının iddiaları ve talepler",
        n_results=2,
        where={"section": "dava_bilgisi"}
    )
    vector_store.print_search_results(results, max_content_length=300)
    
    print("\n" + "="*70)
    print("✅ Vector store hazır! Artık RAG sistemi için kullanılabilir.")
    print("="*70)